<a href="https://colab.research.google.com/github/bhoomiy/My_AIML_Journey/blob/main/text_summarizer/text_summarizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install "transformers[torch]"

In [ ]:
import pandas as pd
from transformers import T5Tokenizer,Trainer,TrainingArguments,T5ForConditionalGeneration

In [ ]:
train_data=pd.read_csv("samsum-train.csv")
val_data=pd.read_csv("samsum-validation.csv")

In [ ]:
train_data.shape

(14732, 3)

In [ ]:
#random sampling(jyada data poitns nhi chahiye hume because training time increase hojayega)
train_data=train_data.sample(n=4000,random_state=42).reset_index(drop=True)
val_data=val_data.sample(n=500,random_state=42).reset_index(drop=True)

In [ ]:
train_data.shape

(4000, 3)

In [ ]:
#Data preprocessing
import re

def clean_data(text):
    text=re.sub(r"\r\n"," ",text) #replace lines
    text=re.sub(r"\s+"," ",text)#remove spaces
    text=re.sub(r"<.*?>"," ",text) #remove html tags
    text=text.strip().lower() #remove trailing spaces and convert everything tolowerspace
    return text

In [ ]:
train_data["dialogue"]=train_data["dialogue"].apply(clean_data)
train_data["summary"]=train_data["summary"].apply(clean_data)

val_data["dialogue"]=val_data["dialogue"].apply(clean_data)
val_data["summary"]=val_data["summary"].apply(clean_data)

In [ ]:
train_data["dialogue"][0]

"violet: hi! i came across this austin's article and i thought that you might find it interesting violet:   claire: hi! :) thanks, but i've already read it. :) claire: but thanks for thinking about me :)"

In [ ]:
#Tokenization
tokenizer=T5Tokenizer.from_pretrained("t5-small")

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

c:\Users\RADHAGOPINATH\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\RADHAGOPINATH\.cache\huggingface\hub\models--t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [ ]:
#raw data->tokens for fine tuning

def tokenize(data):
    inputs=tokenizer(data["dialogue"],padding="max_length",max_length=512,truncation=True)#dialogues ke liye 512 tokens hoge(joh chotte hoge unko padding apply hoga)
    target=tokenizer(data["summary"],padding="max_length",max_length=150,truncation=True)#summary ke liye 150 tokens hoge(--//--)

    inputs["labels"]=target["input_ids"] #token ids=>add to input as labels(target ko inputs se link kr rhe h)
    return inputs


In [ ]:
train_dataset=train_data.apply(tokenize,axis=1).to_list()
val_dataset=val_data.apply(tokenize,axis=1).to_list()

In [ ]:
#working with our model
model=T5ForConditionalGeneration.from_pretrained("t5-small")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
import torch

if torch.backends.mps.is_available():
    device=torch.device("mps")
elif torch.cuda.is_available():
    device=torch.device("cuda")
else:
    device=torch.device("cpu")

print("device: ",device)
model.to(device)


device:  cpu


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [ ]:
#imp training arguements
training_args=TrainingArguments(
    output_dir="./results",
    weight_decay=0.01,
    num_train_epochs=6,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    warmup_steps=500
)

In [ ]:
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [ ]:
#train the model
trainer.train()

In [45]:
model=T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer=T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [ ]:
#test the core logic of summarization
def summarize_dialogues(dialogue):
    dialogue = clean_data(dialogue)

    inputs = tokenizer(  # tokenize
        dialogue,
        padding="max_length",
        max_length=512,
        truncation=True,
        return_tensors="pt"
    )

    inputs = {key: value.to(device) for key, value in inputs.items()}

    model.to(device)
    model.eval()

    with torch.no_grad():
        targets = model.generate(   # generate the summary => token ids
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=150,
            num_beams=4,
            early_stopping=True
        )

    summary = tokenizer.decode(  # decoded our output
        targets[0],
        skip_special_tokens=True
    )

    return summary



In [52]:
test_dialogue = """ 
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""

summary = summarize_dialogues(test_dialogue)

print("Summary: ", summary)

Summary:  ai adoption has significantly increased over the past few years. experts highlight the importance of responsible ai development, including data privacy, security, and long-term societal impact.
